In [1]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from llama_index.core.node_parser import SentenceWindowNodeParser
from llama_index.core import VectorStoreIndex
from llama_index.core import KnowledgeGraphIndex
from llama_index.core import Document
from sentence_transformers import CrossEncoder
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
from llama_index.llms.groq import Groq
from collections import defaultdict
import re

C:\Users\yoush\AppData\Local\Temp\ipykernel_3996\1593299978.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, CSVLoader, WebBaseLoader
c:\Users\yoush\OneDrive\Desktop\3GenAIProj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
url = "https://www.geeksforgeeks.org/artificial-intelligence/artificial-intelligence"

### MULTIPLE SOURCES LOADING

In [3]:
url_loader = WebBaseLoader(url)
pdf_loader = PyPDFLoader("C:\\Users\\yoush\\OneDrive\\Desktop\\3GenAIProj\\data\\SDG.pdf")
csv_loader = CSVLoader("C:\\Users\\yoush\\OneDrive\\Desktop\\3GenAIProj\\data\\mcqs.csv")

In [4]:
url_docs = url_loader.load()
pdf_docs = pdf_loader.load()
csv_docs = csv_loader.load()

In [5]:
url_docs[0].metadata

{'source': 'https://www.geeksforgeeks.org/artificial-intelligence/artificial-intelligence',
 'title': 'Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeks',
 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.',
 'language': 'en'}

### UNIFIED DOCUMENT DB

In [6]:
# Metadata Tagging
for doc in url_docs:
    doc.metadata["source_type"] = "url"

for doc in pdf_docs:
    doc.metadata["source_type"] = "pdf"

for doc in csv_docs:
    doc.metadata["source_type"] = "csv"

In [7]:
# Unified Document DB
all_docs = url_docs + pdf_docs + csv_docs

### CHUNKING

In [8]:
# Cleaning

for doc in all_docs:
    text = doc.page_content
    text = re.sub(r"\s+", " ", text)
    doc.page_content = text.strip()


In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunked_docs = splitter.split_documents(all_docs)

In [10]:
chunked_docs[0].page_content

'Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeksCoursesTutorialsInterview PrepArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsArtificial Intelligence Tutorial | AI TutorialLast Updated : 28 Feb, 2026Artificial Intelligence (AI) refers to the simulation of human intelligence in machines which allows them to think and act like humans. It involves creating algorithms'

### EMBEDDING

In [11]:
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [12]:

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

### GROQ API LOADING

In [13]:
load_dotenv()

True

In [14]:
api_key = os.getenv("GROQ_API_KEY")

In [15]:
Settings.llm = Groq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    api_key=api_key
)

## INDEXING LAYER

#### FAISS DB 

In [16]:
db = FAISS.from_documents(
    chunked_docs,
    embedding 
)
retriever1 = db.as_retriever(
    similarity_top_k=5
)

#### SENTENCE WINDOW INDEX

In [17]:
llama_docs = [Document( text=doc.page_content, metadata=doc.metadata) for doc in chunked_docs]

node_parser = SentenceWindowNodeParser.from_defaults(
    window_size=3,
    window_metadata_key="window",
    original_text_metadata_key="original_text"
)

nodes = node_parser.get_nodes_from_documents(
    llama_docs
)

sentence_index = VectorStoreIndex(nodes)

retriever2 = sentence_index.as_retriever(
    similarity_top_k=5
)

#### GRAPH INDEX

In [18]:
graph_index = KnowledgeGraphIndex.from_documents(
    llama_docs,
    max_triplets_per_chunk=4
)
retriever3 = graph_index.as_retriever(
    include_text=True
)

## RETRIEVERS

In [19]:
query = "What is Artificial Intelligence?"

In [20]:
vector_results = retriever1.invoke(query)
print(vector_results[0].page_content)

Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeksCoursesTutorialsInterview PrepArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsArtificial Intelligence Tutorial | AI TutorialLast Updated : 28 Feb, 2026Artificial Intelligence (AI) refers to the simulation of human intelligence in machines which allows them to think and act like humans. It involves creating algorithms


In [21]:
sentence_results = retriever2.retrieve(query)
print(sentence_results)

[NodeWithScore(node=TextNode(id_='d13104a2-7ac4-4f93-ae4d-458ec3b64e22', embedding=None, metadata={'source': 'https://www.geeksforgeeks.org/artificial-intelligence/artificial-intelligence', 'title': 'Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en', 'source_type': 'url', 'window': 'Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeksCoursesTutorialsInterview PrepArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsArtificial Intelligence Tutorial | AI TutorialLast Updated : 28 Feb, 2026Artificial Intelligence (AI) refers to the simulation of human intelligence in ma

In [22]:
graph_results = retriever3.retrieve(query)

for node in graph_results:
    print(node.text)

No relationships found.


##### Normalize all retrieved results

In [23]:
retrieved_docs = []

# -------- Vector Retriever --------

for rank, doc in enumerate(vector_results, start=1):

    retrieved_docs.append(
        {
            "text": doc.page_content,
            "source": "vector",
            "rank": rank,
            "metadata": doc.metadata,
        }
    )

# -------- Sentence Window Retriever --------

for rank, node in enumerate(sentence_results, start=1):

    retrieved_docs.append(
        {
            "text": node.node.text,
            "source": "sentence_window",
            "rank": rank,
            "metadata": node.node.metadata,
        }
    )

# -------- Graph Retriever --------

for rank, node in enumerate(graph_results, start=1):

    retrieved_docs.append(
        {
            "text": node.node.text,
            "source": "knowledge_graph",
            "rank": rank,
            "metadata": node.node.metadata,
        }
    )


### RECIPROCAL RANK FUSION (RRF)

In [24]:

k = 60

rrf_scores = defaultdict(
    lambda: {
        "score": 0,
        "metadata": None,
        "source": None,
    }
)

for doc in retrieved_docs:

    text = doc["text"]

    rrf_scores[text]["score"] += 1 / (k + doc["rank"])

    rrf_scores[text]["metadata"] = doc["metadata"]

    rrf_scores[text]["source"] = doc["source"]


### Sort by Fusion Score

In [25]:
fused_results = sorted(
    rrf_scores.items(),
    key=lambda x: x[1]["score"],
    reverse=True,
)

#  Keep Top 20 Fusion Results

TOP_FUSION = 20

candidate_docs = fused_results[:TOP_FUSION]

### RERANKING

In [26]:
#  Semantic Reranking

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [
    (query, doc[0]) for doc in candidate_docs
]
scores = reranker.predict(pairs)

#  Combine Reranker Scores

reranked_docs = []

for (text, info), score in zip(candidate_docs, scores):
 reranked_docs.append(

         {

             "text": text,

             "rerank_score": float(score),

             "fusion_score": info["score"],

             "source": info["source"],

             "metadata": info["metadata"],

         }

     )

### FINAL RANKING

In [27]:
reranked_docs = sorted(
     reranked_docs,
     key=lambda x: x["rerank_score"],
     reverse=True,
)

#### TOP 5 CHUNKS

In [28]:
TOP_CONTEXT = 5

top_docs = reranked_docs[:TOP_CONTEXT]

#### PROMPT

In [29]:
context = "\n\n".join( doc["text"] for doc in top_docs )

prompt = f"""
You are an expert AI assistant. Use ONLY the provided context to answer the question. Response to follow: If the user's question is simple and does not require a detailed explanation, then provide a short response in 1-2 lines, Do not explain too much; keep responses short and clear, and include a daily life example.. If user's question is long and require a detailed explanation, then you response: Define the main topic, and divide it into sections such as DEFINITION, DESCRIPTION, EXAMPLE, and the main points represent in a bullets. For each topic you must provide a daily life example that help to understand. If the answer is not present in the context, reply with: "I couldn't find the answer in the provided documents."

==========================
Context
==========================

{context}

==========================
Question
==========================

{query}

==========================
Answer
==========================
"""

#### LLM CALL

In [30]:
response = Settings.llm.complete(prompt)
print(response.text)

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines, allowing them to think and act like humans. For example, virtual assistants like Siri and Alexa use AI to understand voice commands and respond accordingly.


In [31]:
from langchain_groq import ChatGroq

In [32]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=api_key
)

In [33]:
response = llm.invoke(prompt)
print(response.text)

Artificial Intelligence (AI) refers to the simulation of human intelligence in machines, allowing them to think and act like humans. For example, virtual assistants like Siri or Alexa use AI to understand and respond to voice commands.


#### TOP RETRIEVED DOCUMENTS

In [34]:
for i, doc in enumerate(top_docs, start=1):

    print(f"Rank : {i}")

    print(f"Retriever : {doc['source']}")

    print(f"Fusion Score : {doc['fusion_score']:.6f}")

    print(f"Rerank Score : {doc['rerank_score']:.4f}")

    print("Metadata :")

    print(doc["metadata"])

    print("-" * 80)

Rank : 1
Retriever : sentence_window
Fusion Score : 0.016393
Rerank Score : 5.2288
Metadata :
{'source': 'https://www.geeksforgeeks.org/artificial-intelligence/artificial-intelligence', 'title': 'Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeks', 'description': 'Your All-in-One Learning Portal: GeeksforGeeks is a comprehensive educational platform that empowers learners across domains-spanning computer science and programming, school education, upskilling, commerce, software tools, competitive exams, and more.', 'language': 'en', 'source_type': 'url', 'window': 'Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeksCoursesTutorialsInterview PrepArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsArtificial Intelligence Tutorial | AI TutorialLast Updated : 28 Feb, 2026Artificial Intelligence (AI) refers to the simulation of human intelligence in machin

In [35]:
print("=== VECTOR ===")
for doc in retriever1:
    print(doc[:300])
    print("-" * 50)

print("=== SENTENCE ===")
# sentence_results = retriever2.invoke(query)

for node in sentence_results:
    print(node.node.text)
    print("-" * 50)

print("=== GRAPH ===")
# graph_results = retriever3.retrieve(query)

for node in graph_results:
    print(node.get_content()[:300])
    print("-" * 50)

=== VECTOR ===
('name', None)
--------------------------------------------------
('tags', ['FAISS', 'HuggingFaceEmbeddings'])
--------------------------------------------------
('metadata', None)
--------------------------------------------------
('vectorstore', <langchain_community.vectorstores.faiss.FAISS object at 0x0000022B1DE312E0>)
--------------------------------------------------
('search_type', 'similarity')
--------------------------------------------------
('search_kwargs', {})
--------------------------------------------------
=== SENTENCE ===
Artificial Intelligence Tutorial | AI Tutorial - GeeksforGeeksCoursesTutorialsInterview PrepArtificial IntelligenceInterview QuestionsProject IdeasSearch AlgorithmsLocal Search AlgorithmGenerative AIData ScienceMachine LearningDeep LearningML-ProjectsRoboticsArtificial Intelligence Tutorial | AI TutorialLast Updated : 28 Feb, 2026Artificial Intelligence (AI) refers to the simulation of human intelligence in machines which allows them 